# OCULTAR — Tier 2 Fine-Tuning
**Model:** `openai/privacy-filter` → `ocultar/privacy-filter-fr-finance`  
**Goal:** Improve French finance entity detection — organizations (SARL/SAS/SCI), person names, IBAN boundaries.  
**Data:** 200 train / 50 eval samples from `data/fine-tune/french_finance_*.jsonl`

---
**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch

assert torch.cuda.is_available(), "No GPU found — set Runtime > Change runtime type > T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
%pip install -q transformers==4.51.3 tokenizers datasets evaluate seqeval accelerate peft
%pip uninstall -y torchao  # Colab ships 0.10.0 but peft checks for >=0.16.0; uninstall to skip the check

In [ ]:
# ── 3. Clone repo and verify data ─────────────────────────────────────────────
!if [ -d ocultar ]; then cd ocultar && git pull --depth 1 --rebase; else git clone --depth 1 https://github.com/Edu963/ocultar.git; fi

import os, json
TRAIN_FILE = "ocultar/data/fine-tune/french_finance_train.jsonl"
EVAL_FILE  = "ocultar/data/fine-tune/french_finance_eval.jsonl"

train_lines = open(TRAIN_FILE).readlines()
eval_lines  = open(EVAL_FILE).readlines()
print(f"Train samples: {len(train_lines)}")
print(f"Eval  samples: {len(eval_lines)}")

# Show one example
sample = json.loads(train_lines[0])
print("\nSample:")
print("  text    :", sample['text'])
print("  tokens  :", sample['tokens'])
print("  ner_tags:", sample['ner_tags'])

In [ ]:
# ── 4. Label map (must match data_gen_french_finance.py) ─────────────────────
ID2LABEL = {
    0:  'O',
    1:  'B-account_number', 2:  'I-account_number', 3:  'E-account_number', 4:  'S-account_number',
    5:  'B-private_address', 6:  'I-private_address', 7:  'E-private_address', 8:  'S-private_address',
    9:  'B-private_date',    10: 'I-private_date',    11: 'E-private_date',    12: 'S-private_date',
    13: 'B-private_email',   14: 'I-private_email',   15: 'E-private_email',   16: 'S-private_email',
    17: 'B-private_person',  18: 'I-private_person',  19: 'E-private_person',  20: 'S-private_person',
    21: 'B-private_phone',   22: 'I-private_phone',   23: 'E-private_phone',   24: 'S-private_phone',
    25: 'B-private_url',     26: 'I-private_url',     27: 'E-private_url',     28: 'S-private_url',
    29: 'B-secret',          30: 'I-secret',          31: 'E-secret',          32: 'S-secret',
    33: 'B-organization',    34: 'I-organization',    35: 'E-organization',    36: 'S-organization',
    37: 'B-amount',          38: 'I-amount',          39: 'E-amount',          40: 'S-amount',
}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
print(f"Label count: {len(ID2LABEL)}")

In [ ]:
# ── 5. Load tokenizer and model ───────────────────────────────────────────────
# AutoTokenizer fails because tokenizer_config.json references a custom
# "TokenizersBackend" class. PreTrainedTokenizerFast reads tokenizer.json
# directly, bypassing the class lookup — same tokenizer, no error.
from transformers import PreTrainedTokenizerFast, AutoModelForTokenClassification

MODEL_ID = "openai/privacy-filter"

tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_ID)
print(f"Tokenizer: {type(tokenizer).__name__}, vocab size: {tokenizer.vocab_size}")

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(ID2LABEL),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,  # original model has fewer label classes
    trust_remote_code=True,
)
print(f"Model loaded — {sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")

In [ ]:
# ── 6. Load and tokenize dataset ──────────────────────────────────────────────
from datasets import load_dataset
from transformers import DataCollatorForTokenClassification

dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "test": EVAL_FILE})

def tokenize_and_align(examples):
    enc = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )
    all_labels = []
    for i, raw_labels in enumerate(examples["ner_tags"]):
        word_ids = enc.word_ids(batch_index=i)
        prev_word = None
        label_ids = []
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word:
                label_ids.append(raw_labels[wid])
            else:
                label_ids.append(-100)  # sub-tokens ignored
            prev_word = wid
        all_labels.append(label_ids)
    enc["labels"] = all_labels
    return enc

tokenized = dataset.map(tokenize_and_align, batched=True)
collator  = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("Dataset tokenized")

In [ ]:
# ── 7. Metrics ────────────────────────────────────────────────────────────────
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_preds  = [[ID2LABEL[p] for p, l in zip(pred, label) if l != -100] for pred, label in zip(preds, labels)]
    true_labels = [[ID2LABEL[l] for p, l in zip(pred, label) if l != -100] for pred, label in zip(preds, labels)]
    res = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": round(res["overall_precision"], 4),
        "recall":    round(res["overall_recall"],    4),
        "f1":        round(res["overall_f1"],        4),
        "accuracy":  round(res["overall_accuracy"],  4),
    }

In [ ]:
# ── 8. Training (LoRA — fits T4 16 GB) ──────────────────────────────────────
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import torch

# Wrap model with LoRA adapters — only ~0.5% of weights are trainable,
# so gradients fit in VRAM even though the base MoE model is ~14 GB.
lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=8,
    lora_alpha=16,
    target_modules="all-linear",   # PEFT ≥ 0.7 — auto-targets every nn.Linear
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["classifier"],  # always fine-tune the NER head in full
)
model_lora = get_peft_model(model, lora_config)
model_lora.print_trainable_parameters()

OUTPUT_DIR = "/content/privacy-filter-fr-finance"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,               # higher LR suits LoRA adapters
    per_device_train_batch_size=4,    # 16→4 to fit T4; accumulate below
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,    # effective batch size = 16
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,      # recompute activations instead of storing
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model_lora,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# ── 9. Final evaluation ───────────────────────────────────────────────────────
results = trainer.evaluate()
print("\n=== Final Eval ===")
for k, v in results.items():
    print(f"  {k}: {v}")

In [ ]:
# ── 10. Per-entity breakdown ──────────────────────────────────────────────────
# Run the fine-tuned model against the eval set and show per-entity F1
from transformers import pipeline as hf_pipeline
import time

ner = hf_pipeline(
    "token-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0,
)

eval_samples = [json.loads(l) for l in eval_lines]

# Track per-entity hits
from collections import defaultdict
entity_hits   = defaultdict(int)
entity_total  = defaultdict(int)
detected_rows = 0

SENSITIVE_TAGS = {1,2,3,17,19,21,22,23,32,33,35}

for s in eval_samples:
    if not any(t in SENSITIVE_TAGS for t in s['ner_tags']):
        continue
    entities = ner(s['text'])
    if entities:
        detected_rows += 1
    for label, idx in [(s['ner_tags'][i], i) for i in range(len(s['ner_tags'])) if s['ner_tags'][i] in SENSITIVE_TAGS]:
        entity_type = ID2LABEL.get(label, 'UNK').split('-')[-1]
        entity_total[entity_type] += 1

sensitive_rows = sum(1 for s in eval_samples if any(t in SENSITIVE_TAGS for t in s['ner_tags']))

print(f"\n=== Detection Summary ===")
print(f"Rows with detection: {detected_rows}/{sensitive_rows} ({100*detected_rows//sensitive_rows}%)")
print(f"\nEntity counts in eval set:")
for entity, count in sorted(entity_total.items()):
    print(f"  {entity}: {count} tokens")

In [ ]:
# ── 11. Save model ────────────────────────────────────────────────────────────
# Merge LoRA adapters into the base weights → standalone model, no PEFT at inference
merged = trainer.model.merge_and_unload()
merged.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# List files
!ls -lh /content/privacy-filter-fr-finance/

In [ ]:
# ── 12. Download as zip ───────────────────────────────────────────────────────
# Option A: direct zip download via Colab
import shutil
from google.colab import files

zip_path = "/content/privacy-filter-fr-finance"
shutil.make_archive(zip_path, 'zip', zip_path)
files.download(f"{zip_path}.zip")
print("Download started — unzip to ocultar/models/privacy-filter-fr-finance/")

In [ ]:
# ── 12b. (Alternative) Save to Google Drive instead ──────────────────────────
# Uncomment if you prefer Drive over direct download

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree(
#     "/content/privacy-filter-fr-finance",
#     "/content/drive/MyDrive/ocultar-models/privacy-filter-fr-finance",
# )
# print("Saved to Google Drive")

## After download

1. Unzip to `ocultar/models/privacy-filter-fr-finance/`
2. Set the environment variable:
   ```bash
   PRIVACY_FILTER_MODEL_PATH=./models/privacy-filter-fr-finance
   ```
3. Restart the SLM sidecar — it will load the fine-tuned weights automatically:
   ```bash
   docker compose up slm-engine --build
   ```
4. Re-run the benchmark to confirm improvement:
   ```bash
   python3 scripts/benchmark_v2.py
   ```

Target: French finance row detection **≥ 90%** (up from 80%).